# Data quality analysis

### **Import libraries**

In [25]:
import pandas as pd
import numpy as np

### **load raw data**

In [26]:
DATA_PATH = "../data/raw/online_retail_II.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### **Convert Invoice date**

In [27]:
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)

print(df["InvoiceDate"].dtype)

datetime64[ns]


Convert String/text format date into Python understanding Standard Datetime object.

### **Investigate cancelled transactions**

In [28]:
df["IsCancelled"] = (
    df["Invoice"]
    .astype(str)
    .str.startswith("C")
)

df["IsCancelled"].value_counts()

IsCancelled
False    1047877
True       19494
Name: count, dtype: int64

### **Investigate negative quantities**

In [29]:
negative_quantity_df = df[df["Quantity"] < 0]

negative_quantity_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,IsCancelled
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,True
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,True
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,True
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,True
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,True


### **Compare cancellations and negative quantities**

In [30]:
cancellation_analysis = pd.DataFrame({
    "Category": [
        "Cancelled + Negative Quantity",
        "Negative Quantity without Cancellation",
        "Positive Quantity"
    ],
    "Count": [
        (
            df["IsCancelled"] &
            (df["Quantity"] < 0)
        ).sum(),

        (
            ~df["IsCancelled"] &
            (df["Quantity"] < 0)
        ).sum(),

        (df["Quantity"] > 0).sum()
    ]
})

cancellation_analysis

,Category,Count
0,Cancelled + Negative Quantity,19493
1,Negative Quantity without Cancellation,3457
2,Positive Quantity,1044421


### **Investigate negative prices**

In [31]:
negative_price_df = df[df["Price"] < 0]

negative_price_df[
    ["Invoice", "StockCode", "Description",
     "Quantity", "Price", "Customer ID", "Country"]
]

,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,-38925.87,NaN,United Kingdom
825444,A563186,B,Adjust bad debt,1,-11062.06,NaN,United Kingdom
825445,A563187,B,Adjust bad debt,1,-11062.06,NaN,United Kingdom


### **Investigate zero prices**

In [32]:
zero_price_df = df[df["Price"] == 0]

zero_price_df[
    ["Invoice", "StockCode", "Description",
     "Quantity", "InvoiceDate", "Customer ID"]
].head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Customer ID
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,NaN
283,489463,71477,short,-240,2009-12-01 10:52:00,NaN
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,NaN
470,489521,21646,NaN,-50,2009-12-01 11:44:00,NaN
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,NaN
3161,489659,21350,NaN,230,2009-12-01 17:39:00,NaN
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,NaN
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,NaN
3731,489781,84292,NaN,17,2009-12-02 11:45:00,NaN
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,NaN


### **Missing customer id analysis**

In [33]:
missing_customer_df = df[df["Customer ID"].isna()]

missing_customer_df.head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,IsCancelled
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom,False
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom,False
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom,False
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom,False
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom,False
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom,False
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom,False
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom,False
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom,False
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom,False


**Check how many are cancellations:**

In [34]:
missing_customer_analysis = pd.DataFrame({
    "Category": [
        "Missing Customer ID",
        "Missing Customer ID + Cancellation",
        "Missing Customer ID + Positive Quantity"
    ],
    "Count": [
        df["Customer ID"].isna().sum(),

        (
            df["Customer ID"].isna() &
            df["IsCancelled"]
        ).sum(),

        (
            df["Customer ID"].isna() &
            (df["Quantity"] > 0)
        ).sum()
    ]
})

missing_customer_analysis

,Category,Count
0,Missing Customer ID,243007
1,Missing Customer ID + Cancellation,750
2,Missing Customer ID + Positive Quantity,238801


### **Missing description**

In [35]:
missing_description_df = df[df["Description"].isna()]

missing_description_df.head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,IsCancelled
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,False
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,False
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom,False
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom,False
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom,False
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom,False
6378,489882,35751C,NaN,12,2009-12-02 16:22:00,0.0,NaN,United Kingdom,False
6555,489898,79323G,NaN,954,2009-12-03 09:40:00,0.0,NaN,United Kingdom,False
6576,489901,21098,NaN,-200,2009-12-03 09:47:00,0.0,NaN,United Kingdom,False
6581,489903,21166,NaN,48,2009-12-03 09:57:00,0.0,NaN,United Kingdom,False


**Check relationship with zero price:**

In [36]:
(
    df["Description"].isna() &
    (df["Price"] == 0)
).sum()

np.int64(4382)